# Lab 9-14 — Controllable Reasoning with Nemotron-Nano-9B-v2 (Hybrid Mamba-Transformer)

*Colab notebook — part of the Nvidia-Colab-labs track. Labs 1-12 covered GPU/LLM fundamentals with plain transformer models; this is the first lab built around NVIDIA's Nemotron model family, and the first model in the whole track that is *not* a pure transformer.*

**Runtime setup:** `Runtime > Change runtime type > GPU`. Read this before you pick one — unlike almost every other lab in this track, this one does **not** comfortably fit the free T4 (~15 GB VRAM). NVIDIA's own model card lists supported hardware as A10G / A100 / H100-80GB / Jetson AGX Thor — the T4 is conspicuously absent, and the setup cell below explains exactly why (short version: a documented incompatibility between bitsandbytes 4-bit quantization and this model's Mamba-2 blocks means the realistic fallback is a bf16 load, which needs ~19-20 GB). If you have Colab Pro, pick **L4** or **A100**. On the free tier, expect to hit VRAM limits or heavy slowdowns — the setup cell tells you what to watch for.

---

## Why this model is different from every other lab in this track

**`nvidia/NVIDIA-Nemotron-Nano-9B-v2`** is a **hybrid Mamba-2 + Transformer** model — 9B parameters, 128K context, NVIDIA Open Model License (fully open, no gated access, no HF token needed, commercial use permitted). Structurally it is *not* the stack-of-attention-blocks architecture every other model in this track has used. Straight from the model's own `config.json`:

- **56 total layers**: **27 Mamba-2** layers, **25 MLP-only** layers, and just **4 Attention** layers.
- The exact layout lives in a `hybrid_override_pattern` string in the config (`M` = Mamba-2, `*` = Attention, `-` = MLP) — Step 1 pulls this from the *loaded model*, not from marketing copy, and cross-checks it against the actual instantiated module classes.

Its standout capability — and the centerpiece of this lab — is **controllable reasoning budget**: a single system-prompt token, `/think` or `/no_think`, switches the model between "reason step-by-step before answering" and "answer directly," at prompt time, with no retraining or separate checkpoint.

### Papers behind this lab

- **[Mamba (Gu & Dao, 2023)](https://arxiv.org/abs/2312.00752)** — the original selective state-space model (SSM): linear-time sequence modeling via a data-dependent recurrence, the architectural ancestor of everything here.
- **[Mamba-2 (Dao & Gu, 2024)](https://arxiv.org/abs/2405.21060)** — the "SSD" (structured state-space duality) reformulation that made Mamba fast on tensor cores and easy to interleave with attention layers, which is what makes a *hybrid* model like this one practical.
- **[Nemotron-H (NVIDIA, 2025)](https://arxiv.org/abs/2504.03624)** — the hybrid Mamba-Transformer model family this checkpoint is built on.
- **[NVIDIA Nemotron Nano 2 (2025)](https://arxiv.org/abs/2508.14444)** — the paper describing this specific reasoning-controllable 9B model.

### What you'll measure

1. **The real layer composition** of a hybrid model — not a claim, an inspection of the loaded weights.
2. **The output difference** between `/think` and `/no_think` on the identical prompt, including the `<think>...</think>` reasoning wrapper.
3. **The latency and token cost** of turning reasoning on, measured (not assumed) across several prompts.
4. **When reasoning budget is worth spending** in production, and the lever (`max_thinking_tokens`) you'd use to cap it.


In [ ]:
# ============================================================
# Environment & Lab Setup — run this cell first
# ============================================================
#
# --- Dependency research: is mamba-ssm actually required? ---
# Nemotron-Nano-9B-v2 ships CUSTOM modeling code (modeling_nemotron_h.py, loaded via
# trust_remote_code=True — it is not yet a first-class AutoModel architecture the way
# Llama/Qwen are). We read that file before writing this cell. HF's own *built-in*
# Mamba/Mamba2 model classes fall back to a slower pure-PyTorch scan when the
# `mamba_ssm` / `causal_conv1d` CUDA packages aren't installed. This repo's code does
# NOT do that: its gated-RMSNorm class calls `rmsnorm_fn` from
# `mamba_ssm.ops.triton.layernorm_gated` unconditionally, with only a bare
# `raise ImportError(...)` if the package is missing — there is no pure-PyTorch
# fallback path here. That makes mamba-ssm (and causal-conv1d, used by the fast
# conv1d path) HARD requirements for this specific checkpoint, not an optional
# speed-up the way they are for vanilla HF Mamba models.
#
# Installing them compiles CUDA kernels from source and can take 5-15+ minutes on
# Colab, and can fail outright if the pinned torch/CUDA build drifts from what the
# package expects (this has happened repeatedly across mamba-ssm versions on Colab —
# see https://github.com/state-spaces/mamba/issues for the current state). If this
# install fails: restart the runtime, re-run this cell alone, and check that
# `torch.__version__` matches a combination mamba-ssm's wheels support. `--no-build-
# isolation` is important — without it, pip builds mamba-ssm against a fresh isolated
# torch-cpu install instead of the CUDA-enabled torch Colab already has.
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q mamba-ssm causal-conv1d --no-build-isolation  # compiles CUDA kernels -- can take 5-15+ min, see note above

import torch

assert torch.cuda.is_available(), "No GPU detected — go to Runtime > Change runtime type > GPU and rerun."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

# --- VRAM math + the quantization decision ---
# 9B params x 2 bytes (bf16) ~= 18 GB of weights alone, before KV cache / Mamba state.
# A free T4 has ~15 GB total, so bf16 will NOT fit. The obvious fix, following Lab
# 09-07's pattern, is NF4 (BitsAndBytesConfig(load_in_4bit=True, ...)), which usually
# takes a 9B model down to ~5-6 GB. We looked into whether that works HERE specifically
# (not just "does bitsandbytes work in general") — bitsandbytes quantizes by swapping
# every nn.Linear for a Linear4bit module, and multiple independent sources (NVIDIA's
# own NVFP4 quantization writeup for this exact model, community writeups on quantizing
# Nemotron Nano 2) report this swap does not play well with Nemotron-H's Mamba-2
# blocks — the blunt community summary is "you can't load Nemotron-H in 4-bit, you
# can't use QLoRA." NVIDIA's own quantized releases use FP8/NVFP4 checkpoints with
# custom kernels instead of bitsandbytes.
#
# So: Step 1 below TRIES NF4 first anyway (cheap to attempt, and bnb/transformers
# compatibility shifts release to release — it may just work by the time you run
# this) and catches the failure to fall back to a plain bf16 load. Budget ~19-20 GB
# VRAM for that fallback. That's the whole reason this lab's runtime note (top of
# notebook) points you at L4/A10G/A100 instead of the free-tier T4 every other lab in
# this track targets — it's a direct, documented consequence of this being a hybrid
# Mamba architecture rather than a plain transformer, not a corner we cut.
if vram_gb < 18:
    print(f"\nWARNING: only {vram_gb:.1f} GB VRAM detected. This lab's realistic fallback path "
          f"(bf16, ~19-20 GB) will likely NOT fit. NF4 (tried first in Step 1) would fit if it "
          f"works on this architecture -- but per the note above, expect it to fail and fall "
          f"back to bf16. If you have Colab Pro, switch to Runtime > Change runtime type > L4 "
          f"or A100 before continuing.")


## Step 1 — Load Nemotron-Nano-9B-v2 and inspect the hybrid architecture

### Why hybrid Mamba-Transformer, and why it matters for serving

An attention layer's KV cache grows **O(T)** with sequence length T: every new token adds one more key/value pair that every future token must attend back to, so both the memory footprint and the per-token compute of attention scale with how much context you've accumulated. At 128K context, that cache dominates VRAM.

A Mamba-2 layer instead keeps a **fixed-size recurrent state** — updated per token in **O(1)**, independent of how long the sequence has gotten so far. Swap most of a model's attention layers for Mamba and long-context serving gets dramatically cheaper: the state doesn't grow, so neither does per-token latency or memory, at any context length.

So why not go **all-Mamba**? Because a fixed-size state is an information bottleneck: no matter how well-trained, it can't losslessly retain arbitrary facts from 100K tokens back the way attention's exact, unbounded key/value lookup can. Tasks that need precise token-level retrieval or copying from far back in context (needle-in-a-haystack lookups, some multi-hop reasoning, exact in-context recall) tend to degrade on pure-SSM models. NVIDIA's answer, and the standard hybrid-SSM answer generally, is to keep a *small* number of attention layers — here, just 4 out of 56 — so the model gets attention's precise long-range recall where it matters, at a fraction of an all-attention model's KV-cache cost.

### Loading it

We try **NF4** first (fast, cheap to attempt) and fall back to **bf16** if it fails for the reasons discussed in the setup cell.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "nvidia/NVIDIA-Nemotron-Nano-9B-v2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

load_mode = None
try:
    print("Attempting NF4 4-bit load...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        trust_remote_code=True,
        device_map="auto",
    )
    load_mode = "nf4"
    print("NF4 load succeeded.")
except Exception as e:
    print(f"NF4 load failed ({type(e).__name__}: {e})")
    print("Falling back to bf16 (needs ~19-20 GB VRAM -- see setup cell note)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        device_map="auto",
    )
    load_mode = "bf16"

print(f"\nLoaded in mode: {load_mode}")
print(f"Total params: {model.num_parameters()/1e9:.2f}B")


### 🐛 Common mistake

Forgetting `trust_remote_code=True` on **both** the tokenizer and model calls (or only passing it to one). Because this checkpoint ships custom modeling code, `AutoModelForCausalLM.from_pretrained` without it either throws or silently tries to resolve `NemotronHForCausalLM` against transformers' built-in registry, which may not (yet) have it — you'll get a confusing `KeyError`/`ValueError` about an unrecognized architecture instead of a clear "pass trust_remote_code=True" prompt on some transformers versions.

In [ ]:
# Inspect the actual layer composition -- don't take "hybrid Mamba-Transformer" on faith.
from collections import Counter

pattern = model.config.hybrid_override_pattern
print(f"hybrid_override_pattern ({len(pattern)} chars, one per layer):\n  {pattern}\n")

counts = Counter(pattern)
print(f"Mamba-2 layers ('M'):   {counts['M']}")
print(f"Attention layers ('*'): {counts['*']}")
print(f"MLP-only layers ('-'):  {counts['-']}")
print(f"Total: {sum(counts.values())}  (config.num_hidden_layers = {model.config.num_hidden_layers})")

# Cross-check against the classes actually instantiated in the loaded model, rather than
# trusting the config string alone. We don't assume an exact attribute path here (custom
# modeling code varies) -- instead we scan every submodule and bucket by keyword in its
# class name. If the model backs up the config, the class carrying "mamba"/"ssm" in its
# name should show up ~27 times and the attention class ~4 times.
class_counts = Counter(type(m).__name__ for m in model.modules())
mamba_classes = {k: v for k, v in class_counts.items() if "mamba" in k.lower() or "ssm" in k.lower()}
attn_classes  = {k: v for k, v in class_counts.items() if "attention" in k.lower()}
mlp_classes   = {k: v for k, v in class_counts.items() if "mlp" in k.lower()}

print("\nInstantiated module classes containing 'mamba'/'ssm':")
for k, v in sorted(mamba_classes.items()):
    print(f"  {k}: {v}")
print("\nInstantiated module classes containing 'attention':")
for k, v in sorted(attn_classes.items()):
    print(f"  {k}: {v}")
print("\nInstantiated module classes containing 'mlp':")
for k, v in sorted(mlp_classes.items()):
    print(f"  {k}: {v}")

print("\nNote: these are raw submodule counts, not layer counts -- a single decoder layer can "
      "contain several named submodules (e.g. a Mamba mixer plus its own internal projections), "
      "so don't expect these numbers to sum to 56. What matters is that the class carrying the "
      "reasoning-relevant name (the mixer/attention class used once per layer of that type) "
      "tracks the pattern-derived counts above -- Mamba should heavily outnumber Attention.")


## Step 2 — `/think` vs `/no_think`: reasoning on vs off

The reasoning toggle is not a generation parameter — it's a **system-prompt token**. Put `/think` (or nothing — reasoning is on by default) or `/no_think` as the entire content of the `system` message, and the model wraps a reasoning trace in **`<think>...</think>`** tags before its final answer when reasoning is on, and skips straight to the answer when it's off.

Recommended sampling (from NVIDIA's model card / blog post):
- **`/think`**: `temperature=0.6, top_p=0.95`, sampling on. Reasoning traces are verbose — budget `max_new_tokens` in the hundreds to 1000+.
- **`/no_think`**: greedy decoding (`do_sample=False`), and far fewer tokens needed since there's no trace to generate.

We run the *same* prompt — a word problem with a classic "obvious wrong answer" trap (multiply-by-20 gives 100 minutes, which is wrong; the machines run in parallel, so it's still 5 minutes) — through both modes.

In [ ]:
PROMPT = ("Five machines can make five widgets in five minutes. How many minutes would it take "
          "100 machines to make 100 widgets? Show your reasoning.")

def generate(prompt, think, max_new_tokens):
    system_content = "/think" if think else "/no_think"
    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": prompt},
    ]
    tokenized = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
    )
    if think:
        gen_kwargs.update(do_sample=True, temperature=0.6, top_p=0.95)
    else:
        gen_kwargs.update(do_sample=False)  # greedy, per NVIDIA's /no_think recommendation

    out = model.generate(tokenized, **gen_kwargs)
    # Slice off the prompt tokens and keep special tokens so <think>/</think> stay visible.
    return tokenizer.decode(out[0][tokenized.shape[1]:], skip_special_tokens=False)

print("=== /think ===")
think_out = generate(PROMPT, think=True, max_new_tokens=1024)
print(think_out)

print("\n=== /no_think ===")
no_think_out = generate(PROMPT, think=False, max_new_tokens=256)
print(no_think_out)


### 🐛 Common mistake

Decoding with `skip_special_tokens=True`. That's the right default for a clean user-facing answer, but it silently deletes `<think>`/`</think>` along with real special tokens (EOS, pad), so the reasoning trace becomes invisible and you can't tell the two modes apart from the printed text alone. Decode with `skip_special_tokens=False` any time you actually want to see or parse the reasoning wrapper — as we do below.

A second common mistake: decoding `out[0]` directly instead of `out[0][tokenized.shape[1]:]`. Without the slice you re-print the entire prompt (system + user turns) before the new generation, which is confusing and makes latency-per-output-token math wrong in Step 3.

In [ ]:
import re

def split_reasoning(text):
    '''Split a decoded generation into (reasoning_trace, final_answer).

    Handles two chat-template behaviours we can't be 100% sure of ahead of time without
    running this on real hardware: some reasoning-model templates emit the *opening*
    <think> tag themselves as part of the generated tokens, others pre-seed <think>\n
    into the prompt (so only the closing </think> shows up in what we decode here).
    '''
    if "<think>" in text and "</think>" in text:
        m = re.search(r"<think>(.*?)</think>(.*)", text, re.DOTALL)
        return m.group(1).strip(), m.group(2).strip()
    elif "</think>" in text:
        reasoning, answer = text.split("</think>", 1)
        return reasoning.strip(), answer.strip()
    else:
        return None, text.strip()

reasoning, answer = split_reasoning(think_out)
print("--- /think: reasoning trace ---")
print(reasoning if reasoning else "(no <think> tag found in decoded output -- print tokenizer.chat_template "
                                   "to check how this model version wraps reasoning)")
print("\n--- /think: final answer ---")
print(answer)

_, no_think_answer = split_reasoning(no_think_out)
print("\n--- /no_think: output (should go straight to the answer, no trace) ---")
print(no_think_answer)


## Step 3 — Quality vs latency tradeoff sweep

Reasoning isn't free — every token of "thinking" is a token you generate (and pay for, and wait for) before the model gives you an answer. We make that concrete by running a small set of reasoning-friendly prompts (math + logic) through both modes and measuring **wall-clock generation time** and **output token count**, the same benchmarking pattern used in Lab 09-04's precision sweep (warmup-free here since we only run once per prompt/mode, but still `torch.cuda.synchronize()` around the timer so we're timing actual GPU completion, not just kernel submission).

In [ ]:
import time

SWEEP_PROMPTS = [
    "A farmer has chickens and rabbits. Together they have 35 heads and 94 legs. How many chickens and how many rabbits are there?",
    "If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?",
    "Three friends split a bill. Alice paid $12 more than Bob, and Carol paid twice what Bob paid. The total bill was $84. How much did each person pay?",
    "All bloops are razzles. All razzles are lazzles. Are all bloops definitely lazzles? Explain your reasoning step by step.",
    "A is twice as old as B was when A was as old as B is now. If A is 30 years old today, how old is B?",
]

def timed_generate(prompt, think, max_new_tokens):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    text = generate(prompt, think=think, max_new_tokens=max_new_tokens)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    n_tokens = len(tokenizer(text, add_special_tokens=False).input_ids)
    return elapsed, n_tokens

sweep_results = []
for i, p in enumerate(SWEEP_PROMPTS):
    print(f"[{i+1}/{len(SWEEP_PROMPTS)}] {p[:60]}...")
    t_think, n_think = timed_generate(p, think=True, max_new_tokens=768)
    t_no, n_no = timed_generate(p, think=False, max_new_tokens=256)
    sweep_results.append({
        "prompt":           p[:40] + "...",
        "think_seconds":    t_think,
        "think_tokens":     n_think,
        "no_think_seconds": t_no,
        "no_think_tokens":  n_no,
    })
    print(f"    /think:    {t_think:6.1f}s, {n_think:4d} output tokens")
    print(f"    /no_think: {t_no:6.1f}s, {n_no:4d} output tokens")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = [f"P{i+1}" for i in range(len(sweep_results))]
think_times = [r["think_seconds"] for r in sweep_results]
no_think_times = [r["no_think_seconds"] for r in sweep_results]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, think_times, width, label="/think")
ax.bar(x + width/2, no_think_times, width, label="/no_think")
ax.set_ylabel("Wall-clock generation time (s)")
ax.set_xlabel("Prompt")
ax.set_title("Reasoning cost: /think vs /no_think latency across prompts")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
plt.tight_layout()
plt.show()

avg_think = sum(think_times) / len(think_times)
avg_no_think = sum(no_think_times) / len(no_think_times)
avg_think_tok = sum(r["think_tokens"] for r in sweep_results) / len(sweep_results)
avg_no_think_tok = sum(r["no_think_tokens"] for r in sweep_results) / len(sweep_results)

print(f"\nAverage /think:    {avg_think:6.1f}s, {avg_think_tok:5.0f} output tokens")
print(f"Average /no_think: {avg_no_think:6.1f}s, {avg_no_think_tok:5.0f} output tokens")
print(f"/think is {avg_think/max(avg_no_think, 1e-6):.1f}x slower and emits "
      f"{avg_think_tok/max(avg_no_think_tok, 1e-6):.1f}x more tokens than /no_think on this prompt set.")
print("\nThat multiplier is the real production cost of reasoning: more tokens generated, more "
      "wall-clock latency, more $ at inference-provider pricing that bills by output token.")


## Step 4 — When to use reasoning mode in production

The Step 3 numbers make the tradeoff concrete: `/think` costs real tokens and real wall-clock time for a real accuracy benefit on multi-step problems. The production decision is which side of that tradeoff a given request type sits on.

| Use `/think` when... | Use `/no_think` when... |
|---|---|
| Multi-step math, logic, or planning where an early mistake compounds | Simple factual lookups or classification (intent routing, sentiment, tagging) |
| Agentic tool-call sequencing — deciding *which* tools, in *what order* | Latency-sensitive chat / autocomplete-style interactions |
| Ambiguous requests that benefit from the model working through constraints out loud | High-QPS endpoints where output-token cost dominates your bill |
| Anything you'd want a human to "show their work" on before you trust the answer | You already know the answer requires no derivation (retrieval, formatting, short replies) |

In practice, most production traffic is a *mix* of both request types, and hard-coding `/think` vs `/no_think` per endpoint is a blunt instrument. NVIDIA's vLLM deployment path exposes a **`max_thinking_tokens`** parameter that caps the reasoning trace length per request — instead of an all-or-nothing switch, you get a dial: let the model reason, but insert `</think>` once it hits the budget, forcing it to commit to an answer. NVIDIA reports this can cut inference cost by up to ~60% with little accuracy loss on many workloads, by capping the (small fraction of) requests where reasoning would otherwise run unnecessarily long. This lab doesn't re-implement vLLM serving — Labs **09-02** and **09-05** already cover the vLLM serving mechanics in depth; `max_thinking_tokens` is simply another request-level knob you'd set alongside everything covered there.

---

## What you just built

- Loaded a genuinely different architecture family from the rest of this track — a **hybrid Mamba-2 + Transformer** model, not a stack of attention blocks — and confirmed its real layer composition (27 Mamba-2 / 25 MLP / 4 Attention) by inspecting the loaded model, not by trusting the README.
- Exercised the model's built-in **reasoning-budget control** (`/think` vs `/no_think`) as a system-prompt toggle, and looked inside the `<think>...</think>` wrapper instead of treating it as a black box.
- **Measured, not assumed,** the latency and token cost of turning reasoning on, across a small prompt sweep — turning "reasoning costs tokens" from a talking point into a number.
- Investigated a real dependency/quantization risk (mamba-ssm's hard requirement, bitsandbytes' incompatibility with Mamba-2 blocks) *before* writing the setup cell, instead of discovering it the hard way at `pip install` time.

## What to read next

- **[NVIDIA-Nemotron-Nano-9B-v2 model card](https://huggingface.co/nvidia/NVIDIA-Nemotron-Nano-9B-v2)** — the source of truth for config fields, license, and supported hardware used throughout this lab.
- **[Supercharge AI Reasoning with Nemotron Nano 2 (HF blog)](https://huggingface.co/blog/nvidia/supercharge-ai-reasoning-with-nemotron-nano-2)** — more detail on the `/think`/`/no_think` mechanism and `max_thinking_tokens` budget control.
- **[Nemotron-H architecture docs](https://docs.nvidia.com/nemo/megatron-bridge/0.3.1/models/llm/nemotronh.html)** — NVIDIA's own writeup of the hybrid layer pattern and design rationale.
- **[Mamba (Gu & Dao, 2023)](https://arxiv.org/abs/2312.00752)** and **[Mamba-2 (Dao & Gu, 2024)](https://arxiv.org/abs/2405.21060)** — background reading on the SSM architecture underneath all of this.
- **[Nemotron-H paper (2025)](https://arxiv.org/abs/2504.03624)** and **[Nemotron Nano 2 paper (2025)](https://arxiv.org/abs/2508.14444)** — the primary sources behind the architecture and reasoning-control claims in this lab.

## What to try next

- Re-run Step 2 with `do_sample=True` on `/no_think` and compare determinism against the greedy default.
- Approximate `max_thinking_tokens` yourself: regenerate with a custom `StoppingCriteria` that forces `</think>` once you hit a token budget mid-reasoning, and see how answer quality degrades as you shrink the budget.
- Probe the 4-attention-layer hypothesis directly: feed a long passage and ask for exact retrieval ("what is the 4th word after 'X'?") to see whether this hybrid model still nails needle-in-haystack-style lookups the way a full-attention model would.
- If you have A100/H100 access, try a larger Nemotron-H checkpoint and re-run the Step 3 sweep — see whether the /think-vs-/no_think latency ratio holds steady or shifts with model size.

## What we could not verify by actually running this

This build environment has no GPU, so **none of the code above was executed end-to-end** — every design choice here comes from reading the model card, its linked `config.json` / `modeling_nemotron_h.py`, NVIDIA's architecture docs, and community reports, not from a real run. Specific things to watch for on your first execution:

1. **Whether `pip install mamba-ssm causal-conv1d --no-build-isolation` actually succeeds** on whatever Colab image you get. CUDA/torch pinning on Colab drifts over time, and mamba-ssm's compiled wheels are sensitive to exact version matches — this has been a recurring, actively-discussed pain point (see `github.com/state-spaces/mamba` issues). If it fails, you may need to pin a specific `mamba-ssm==` version compatible with Colab's current torch/CUDA.
2. **Whether the NF4 `try/except` in Step 1 fails cleanly.** We're fairly confident, from multiple independent sources, that bitsandbytes 4-bit is incompatible with Nemotron-H's Mamba-2 blocks — but we don't know whether that surfaces as a catchable Python exception (as the `except Exception` block assumes) or as something messier, like a partial load followed by a CUDA device-side assert, or a silent wrong-output failure that doesn't raise at all. If NF4 "succeeds" but generation output looks garbled, that's the likely culprit — skip straight to the bf16 branch.
3. **The exact `<think>...</think>` tag behavior of the chat template**, including whether the opening `<think>` tag is emitted as a generated token or pre-seeded into the prompt before generation starts. `split_reasoning()` in Step 2 handles both cases, but if it still returns `None` for the reasoning trace, print `tokenizer.chat_template` and inspect it directly — model-card revisions occasionally change this detail.
4. **Actual VRAM headroom for the bf16 fallback.** ~19-20 GB is our estimate from param count × 2 bytes plus overhead; real KV-cache and Mamba-state memory at 1024+ generated tokens (and this model's 128K context capability) could push the requirement higher than that back-of-envelope number.
5. **Generation speed if mamba-ssm's Triton kernels don't fire well on your specific GPU.** T4 is old enough that some Triton kernel paths aren't as well-optimized for it as for Ampere+/Hopper — if generation is dramatically slower than expected, that's consistent with either the kernels not engaging cleanly or (if the mamba-ssm install failed silently in some way) an unexpectedly slow code path. Budget real wall-clock slack the first time you run Step 3's sweep.
